# SILICON — render the 15 s film in Colab

Same five scripts the GitHub Actions workflow runs, no billing required. A free T4 renders this on
Cycles GPU in a few minutes; CPU works too, just slower.

**One manual step:** upload the `blender-film/` folder from the repo into `/content` (Files pane →
drag the folder in). All five files must be there: `film_config.py`, `scene.py`, `vo.py`,
`score.py`, `gates.py`.

Runtime → Change runtime type → T4 GPU first.

In [ ]:
!apt-get -qq update
!DEBIAN_FRONTEND=noninteractive apt-get -qq install -y libxrender1 libxi6 libxfixes3 \
    libxxf86vm1 libsm6 libice6 libxt6 libgl1 libegl1 libgomp1 ffmpeg 2>&1 | tail -2
!pip -q install bpy==5.0.1 kokoro-onnx numpy
print("deps ok")

In [ ]:
import os, subprocess
os.makedirs("/content/kokoro", exist_ok=True)
base = "https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0"
for f in ("kokoro-v1.0.onnx", "voices-v1.0.bin"):
    if not os.path.exists(f"/content/kokoro/{f}"):
        subprocess.run(["curl", "-sSL", "--retry", "3", "-o", f"/content/kokoro/{f}", f"{base}/{f}"], check=True)
print(subprocess.run(["ls", "-lh", "/content/kokoro"], capture_output=True, text=True).stdout)

In [ ]:
# 1. voice-over  (Kokoro, ~1 min on CPU)
!cd /content && python3 blender-film/vo.py --out blender-film/out/vo.wav \
    --model /content/kokoro/kokoro-v1.0.onnx --voices /content/kokoro/voices-v1.0.bin

# 2. score + mix, normalised to -16 LUFS
!cd /content && python3 blender-film/score.py --vo blender-film/out/vo.wav --out blender-film/out/mix.wav
!cd /content && ffmpeg -y -v error -i blender-film/out/mix.wav \
    -af "loudnorm=I=-16:TP=-1.0:LRA=11" -ar 48000 -ac 2 blender-film/out/soundtrack.wav && echo "soundtrack ok"

In [ ]:
# 3. picture — smoke test first: builds the scene and renders 5 frames at half res
import subprocess
r = subprocess.run(["python3", "blender-film/scene.py", "--", "--preset", "preview",
                    "--out", "blender-film/out", "--smoke", "1,90,200,340,445"],
                   cwd="/content", capture_output=True, text=True)
lines = [l for l in (r.stdout + r.stderr).splitlines()
         if any(k in l for k in ("[scene]", "[smoke]", "Error", "Traceback"))]
print("\n".join(lines[-20:]))
from IPython.display import Image, display
for f in sorted(os.listdir("/content/blender-film/out/smoke"))[:5]:
    display(Image(filename=f"/content/blender-film/out/smoke/{f}", width=380))

In [ ]:
# 4. full preview pass (540p, 12 spp). Change preset to "final" for 1080p / 64 spp.
import time, subprocess
t0 = time.time()
r = subprocess.run(["python3", "blender-film/scene.py", "--", "--preset", "preview",
                    "--out", "blender-film/out"], cwd="/content", capture_output=True, text=True)
print((r.stdout or "")[-800:], (r.stderr or "")[-400:])
print(f"render wall time: {(time.time()-t0)/60:.1f} min")

In [ ]:
# 5. encode + gates
!cd /content && ffmpeg -y -v error -framerate 30 -start_number 1 \
    -i blender-film/out/frames_preview/f_%04d.png -i blender-film/out/soundtrack.wav \
    -c:v libx264 -preset slow -crf 18 -pix_fmt yuv420p -profile:v high \
    -x264-params "aq-mode=3:deblock=-1,-1" -c:a aac -b:a 256k -ar 48000 \
    -movflags +faststart blender-film/out/silicon_preview.mp4 && ls -lh blender-film/out/silicon_preview.mp4

!cd /content && python3 blender-film/gates.py --video blender-film/out/silicon_preview.mp4 \
    --audio blender-film/out/soundtrack.wav --manifest blender-film/out/manifest_preview.json \
    --json blender-film/out/gates_preview.json

from IPython.display import Video
Video("/content/blender-film/out/silicon_preview.mp4", embed=True, width=720)

### If a gate fails

The gates are the point — they measure the file, not the intent. Typical first-run failures and what
they mean:

- `mean_saturation` high → the palette drifted bright; reduce light energy, not the gate.
- `type_resolved` low → a text window had no crisp frames: the type is out of shot, too dark, or
  blurred past recognition at that depth.
- `cuts_detected` low → a cut isn't landing as a discontinuity; check the marker cameras and the
  last-frame push before it.
- `voice_over_bed_db` low → speech buried under the score: raise `--bed` in `score.py`, or the mix
  is being normalised too hard.

Download `blender-film/out/silicon_preview.mp4` when it's done — the file survives only until the
Colab VM recycles.